# 01 — Data Exploration

Explores the three raw MovieLens 32M CSVs:
- `movies.csv`  — 87,585 movies (movieId, title, genres)
- `ratings.csv` — 32 M ratings (userId, movieId, rating, timestamp)
- `tags.csv`    — 2 M user tags (userId, movieId, tag, timestamp)

**Optimisations applied**
- Dtype hints on `ratings` cut memory ~488 MB → ~244 MB (`int32` / `float32`).
- Dropped `timestamp` columns immediately — not used downstream.
- `usecols` avoids loading unused columns from tags.
- Histogram x-axis limited to remove long-tail blank space.
- Tags encoded with `encoding='utf-8', errors='replace'` to handle garbled bytes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ── Load datasets ─────────────────────────────────────────────────────────────
# dtype hints halve ratings memory (int64→int32, float64→float32).
# usecols drops timestamp at read time — saves one full column across 32 M rows.

movies = pd.read_csv(
    "../data/movies.csv",
    dtype={"movieId": "int32"},
    encoding="utf-8",
)

ratings = pd.read_csv(
    "../data/ratings.csv",
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32"},
    usecols=["userId", "movieId", "rating"],
    encoding="utf-8",
)

tags = pd.read_csv(
    "../data/tags.csv",
    dtype={"movieId": "int32"},
    usecols=["userId", "movieId", "tag"],
    encoding="utf-8",
    errors="replace",   # replace garbled non-UTF-8 bytes instead of crashing
)

print(f"movies  : {movies.shape}")
print(f"ratings : {ratings.shape}  |  memory: {ratings.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"tags    : {tags.shape}")

In [ ]:
movies.head()

In [ ]:
ratings.head()

In [ ]:
tags.head()

In [ ]:
movies.info()

In [ ]:
ratings.info()

In [ ]:
tags.info()

## Missing Values Analysis

In [ ]:
print("movies :", movies.isnull().sum().to_dict())
print("ratings:", ratings.isnull().sum().to_dict())
print("tags   :", tags.isnull().sum().to_dict())

In [ ]:
# Drop the 17 rows with a null tag (0.00085 % of the tags dataset).
tags_clean = tags.dropna(subset=["tag"]).copy()
print(f"Tags after dropping nulls: {len(tags_clean):,}  (dropped {len(tags) - len(tags_clean)})")

## User Behavior Analysis

In [ ]:
print(f"Unique Users  : {ratings['userId'].nunique():,}")
print(f"Unique Movies : {ratings['movieId'].nunique():,}")
print(f"Total Ratings : {len(ratings):,}")

In [ ]:
# Rating distribution — figsize set explicitly for readability.
fig, ax = plt.subplots(figsize=(8, 4))
ratings["rating"].hist(bins=10, ax=ax)
ax.set_title("Distribution of Ratings")
ax.set_xlabel("Rating")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
user_activity = ratings.groupby("userId").size()
user_activity.describe()

In [ ]:
# Cap x-axis at 2000 so the long-tail doesn't crush the visible bars.
fig, ax = plt.subplots(figsize=(10, 4))
user_activity.clip(upper=2000).hist(bins=50, ax=ax)
ax.set_title("Ratings per User  (capped at 2,000 for readability)")
ax.set_xlabel("Number of Ratings")
ax.set_ylabel("Users")
plt.tight_layout()
plt.show()

## Top Movies by Rating Count

In [ ]:
movie_popularity = (
    ratings
    .groupby("movieId", sort=False)
    .size()
    .reset_index(name="num_ratings")
)
popular_movies = movie_popularity.merge(movies[["movieId", "title", "genres"]], on="movieId")
popular_movies.sort_values("num_ratings", ascending=False).head(20)

## Highest-Rated Movies  (min 100 ratings)

In [ ]:
movie_stats = (
    ratings
    .groupby("movieId", sort=False)
    .agg(avg_rating=("rating", "mean"), num_ratings=("rating", "count"))
)
movie_stats = movie_stats[movie_stats["num_ratings"] >= 100]
movie_stats.sort_values("avg_rating", ascending=False).head(20)

## Genre Distribution

In [ ]:
genre_counts = movies["genres"].str.split("|").explode().value_counts()
genre_counts

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
genre_counts.plot.bar(ax=ax)
ax.set_title("Movie Genre Distribution")
ax.set_xlabel("Genre")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Sparsity Analysis

In [ ]:
n_users  = ratings["userId"].nunique()
n_movies = ratings["movieId"].nunique()

sparsity = 1 - len(ratings) / (n_users * n_movies)
print(f"Sparsity: {sparsity:.2%}")